In this notebook, the files loaded into the volumes are read and stored into tables with a defined schema.

In [0]:
# Volume base path where CSV files are stored
volume_base_path = '/Volumes/Data_Lakehouse_Databricks/Bronze/Bronze_Vol/'

# Catalog and schema for bronze tables
catalog = 'Data_Lakehouse_Databricks'
schema = 'Bronze'

# Folders containing CSV files
folders = ['source_erp', 'source_crm']

print(f"Creating bronze tables in {catalog}.{schema}...\n")

for folder in folders:
    folder_path = volume_base_path + folder + '/'
    print(f"Processing folder: {folder}")
    
    try:
        # List all CSV files in the folder
        files = dbutils.fs.ls(folder_path)
        csv_files = [f for f in files if f.name.endswith('.csv')]
        
        # Process each CSV file
        for file_info in csv_files:
            # Extract filename without extension
            filename = file_info.name.replace('.csv', '')
            # Extract prefix from folder name (source_erp -> erp, source_crm -> crm)
            folder_prefix = folder.replace('source_', '')
            table_name = f"bronze_{folder_prefix}_{filename.lower()}"
            
            print(f"  Reading {file_info.name}...")
            
            # Read CSV with schema inference
            df = spark.read \
                .option("header", "true") \
                .option("inferSchema", "true") \
                .csv(file_info.path)
            
            # Show schema and sample data
            print(f"    Schema for {filename}:")
            df.printSchema()
            print(f"    Rows: {df.count()}")
            
            # Create fully qualified table name
            full_table_name = f"{catalog}.{schema}.{table_name}"
            
            # Write as Delta table
            df.write \
                .format("delta") \
                .mode("overwrite") \
                .saveAsTable(full_table_name)
            
            print(f"    ✓ Created table: {full_table_name}\n")
            
    except Exception as e:
        print(f"  ✗ Error processing {folder}: {str(e)}\n")

print("=== Bronze tables creation completed ===")
